In [2]:
import torch
import torch.nn as nn
import math
import torchvision
print(torch.backends.mps.is_built())

True


# 位置编码与词嵌入

In [3]:
from typing import Any


class Embedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model

    def forward(self, x): #x.shape = (batch_size, seq_len)
        return self.embedding(x) * math.sqrt(self.d_model) #shape = (batch_size, seq_len, d_model)
    
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model) #shape = (max_len, d_model)
        posIDx = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) #shape = (max_len, 1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) #shape = (d_model/2,)

        # 每隔一列使用sin函数，每隔一列使用cos函数
        pe[:, 0::2] = torch.sin(posIDx * div_term) #shape = (max_len, d_model/2)
        pe[:, 1::2] = torch.cos(posIDx * div_term) #shape = (max_len, d_model/2)
        
        # 将计算好的位置编码 pe 注册为模型的缓冲区（buffer），并增加一个批次维度。用于注册不参与梯度更新但需要随模型保存和移动的张量。
        self.register_buffer('pe', pe.unsqueeze(0)) #shape = (1, max_len, d_model)

    def forward(self, x): #x.shape = (batch_size, seq_len, d_model)
        return x + self.pe[:, :x.size(1), :] #shape = (batch_size, seq_len, d_model)


# 多头注意力机制

In [4]:
# mask 形状：应可 broadcast 到 (batch_size, num_heads, seq_len_q, seq_len_k)；
# Padding mask 用 `(batch,1,1,seq_k)` 表示“不同样本的 padding 位置不同，但所有查询位置和注意力头共享同一套 padding 标记”；Causal mask 用 `(1,1,seq_len,seq_len)` 表示“所有样本和头共享同一个下三角规则矩阵”。

def scale_dot_product_attention(q, k, v, mask=None, drop_layer=None):
    d_k = q.size(-1) # d_k=d_model/num_heads
    # 只转置最后两维，是为了将 k 的向量维度 (d_k) 与 q 的向量维度对齐进行点积，同时将 k 的序列长度 (seq_k) 保留为结果矩阵的列，从而得到一个 [seq_q x seq_k] 的注意力分数矩阵。
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k) # shape = (batch_size, num_heads, seq_len_q, seq_len_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf')) 

    # 对每个q的k进行softmax，得到权重矩阵，表示每个q对所有k的注意力分布。
    weight = torch.softmax(scores, dim=-1) # shape = (batch_size, num_heads, seq_len_q, seq_len_k)

    if drop_layer is not None:
        weight = drop_layer(weight)

    return torch.matmul(weight, v) # shape = (batch_size, num_heads, seq_len_q, d_k)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, drop_rate=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_k = d_model // num_heads
        self.num_heads = num_heads

        self.linears = nn.ModuleList([nn.Linear(d_model,d_model) for _ in range(3)]) # 3个线性层分别用于 q, k, v 的线性变换
        self.output_linear = nn.Linear(d_model, d_model) # 最后的线性层用于将多头注意力的输出映射回 d_model 维度
        self.dropout = nn.Dropout(drop_rate)

    def forward(self, q, k, v, mask=None): # shape = (batch_size, seq_len, d_model)
        # 这里的输入的q\k\v其实都是输入X
        batch_size = q.size(0)
        # (batch_sze, seq_len, d_model) -> (batch_size, seq_len, num_heads * d_k) -> (batch_size, seq_len, num_heads, d_k) -> (batch_size, num_heads, seq_len, d_k)
        q = self.linears[0](q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = self.linears[1](k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = self.linears[2](v).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # 计算多头注意力
        x = scale_dot_product_attention(q, k, v, mask, self.dropout) # shape = (batch_size, num_heads, seq_len, d_k)
        # 将多头注意力的输出重新组合成 (batch_size, seq_len, d_model)
        x = x.contiguous().view(batch_size, -1, self.num_heads * self.d_k) 

        return self.output_linear(x) # shape = (batch_size, seq_len, d_model)
    

# 前馈神经网路

In [5]:
class Feedforward(nn.Module):
    def __init__(self, d_model, d_ff, drop_rate=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff) # 第一个线性层将输入从 d_model 维度映射到更高的 d_ff 维度
        self.linear2 = nn.Linear(d_ff, d_model) # 第二个线性层将输出映射回 d_model 维度
        self.dropout = nn.Dropout(drop_rate)

    def forward(self, x): # shape = (batch_size, seq_len, d_model)
        x = self.dropout(torch.relu(self.linear1(x))) # 先通过第一个线性层和 ReLU 激活函数，然后应用 dropout
        return self.linear2(x) # 最后通过第二个线性层输出，shape = (batch_size, seq_len, d_model)
    

# Encoder Layer

In [6]:
class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, drop_rate=0.1):
        super().__init__()
        self.multi_head_attention = MultiHeadAttention(d_model, num_heads, drop_rate)
        self.feed_forward = Feedforward(d_model, d_ff, drop_rate)
        self.norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)]) # 两个 LayerNorm 分别用于多头注意力和前馈网络的残差连接
        self.dropout = nn.Dropout(drop_rate)

    def forward(self, x, mask): # shape = (batch_size, seq_len, d_model)
        x_norm = self.norm[0](x)
        atten_output = self.multi_head_attention(x_norm, x_norm, x_norm, mask) # shape = (batch_size, seq_len, d_model)
        x = x + self.dropout(atten_output) # 残差连接和 dropout
        x = x + self.dropout(self.feed_forward(self.norm[1](x))) # 前馈网络的残差连接和 dropout

        return x # shape = (batch_size, seq_len, d_model)


# Dncoder Layer

In [7]:
class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, drop_rate=0.1):
        super().__init__()
        self.masked_attention = MultiHeadAttention(d_model, num_heads, drop_rate) # 用于解码器自注意力的多头注意力层
        self.cross_attention = MultiHeadAttention(d_model, num_heads, drop_rate) # 用于解码器与编码器输出之间的多头注意力层
        self.feed_forward = Feedforward(d_model, d_ff, drop_rate) # 前馈网络层
        self.norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(3)]) # 三个 LayerNorm 分别用于解码器自注意力、交叉注意力和前馈网络的残差连接
        self.dropout = nn.Dropout(drop_rate)

    def forward(self, x, enc_output, src_mask, tgt_mask): # shape = (batch_size, seq_len, d_model)
        x_norm = self.norm[0](x)
        # tgt_mask is causal + padding
        masked_attention = self.masked_attention(x_norm, x_norm, x_norm, tgt_mask) # 解码器自注意力，shape = (batch_size, seq_len, d_model)
        x = x + self.dropout(masked_attention) # 解码器自注意力的残差连接和 dropout
        cross_attention = self.cross_attention(self.norm[1](x), enc_output, enc_output, src_mask) # 解码器与编码器输出之间的交叉注意力，shape = (batch_size, seq_len, d_model)
        x = x + self.dropout(cross_attention) # 交叉注意力的残差连接和 dropout
        x = x + self.dropout(self.feed_forward(self.norm[2](x))) # 前馈网络的残差连接和 dropout

        return x # shape = (batch_size, seq_len, d_model)

# 完整Transformer

In [8]:
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=512, num_heads=8, num_layers=6, d_ff=2048, dropout=0.1):
        super().__init__()
        self.src_embedding = Embedding(src_vocab, d_model) # 输入嵌入层
        self.tgt_embedding = Embedding(tgt_vocab, d_model) # 输出嵌入层
        self.positional_encoding = PositionalEncoding(d_model) # 位置编码层

        self.encoders = nn.ModuleList([Encoder(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]) # 编码器层列表
        self.decoders = nn.ModuleList([Decoder(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]) # 解码器层列表
        self.encoder_norm = nn.LayerNorm(d_model) # 编码器输出的 LayerNorm
        self.decoder_norm = nn.LayerNorm(d_model) # 解码器输出的 LayerNorm

        self.output_linear = nn.Linear(d_model, tgt_vocab) # 输出线性层，用于将解码器输出映射到目标词汇表大小的维度
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, tgt, src_mask, tgt_mask):
        # encoder
        enc_emb = self.positional_encoding(self.src_embedding(src)) # 输入嵌入加位置编码，shape = (batch_size, seq_len, d_model)
        enc_out = enc_emb
        for encoder in self.encoders:
            enc_out = encoder(enc_out, src_mask) # 编码器层的前向传播，shape = (batch_size, seq_len, d_model)
        enc_out = self.encoder_norm(enc_out) # 编码器输出的 LayerNorm，

        # decoder
        dec_emb = self.positional_encoding(self.tgt_embedding(tgt)) # 输出�嵌入加位置编码，shape = (batch_size, seq_len, d_model)
        dec_out = dec_emb
        for decoder in self.decoders:
            dec_out = decoder(dec_out, enc_out, src_mask, tgt_mask) # 解码器层的前向传播，shape = (batch_size, seq_len, d_model)
        dec_out = self.decoder_norm(dec_out) # 解码器输出的 LayerNorm

        logits = self.output_linear(dec_out) # 输出线性层，shape = (batch_size, seq_len, tgt_vocab)
        return logits

# Mask生成函数

In [9]:
def create_padding_mask(seq, pad_token_id=0):
    return (seq != pad_token_id).unsqueeze(1).unsqueeze(2) # shape = (batch_size, 1, 1, seq_len)

def create_causal_mask(seq_len):
    mask = torch.tril(torch.ones((seq_len, seq_len), dtype=torch.bool)) # shape = (seq_len, seq_len)
    return mask.unsqueeze(0).unsqueeze(0) # shape = (1, 1, seq_len, seq_len)

# 测试

In [10]:

if __name__ == "__main__":
    # example parameters
    src_vocab = 10000
    tgt_vocab = 10000
    model = Transformer(src_vocab, tgt_vocab)

    # assuming input
    batch_size = 2
    src_seq_len = 5
    tgt_seq_len = 4
    src = torch.randint(1, src_vocab, (batch_size, src_seq_len)) 
    tgt = torch.randint(1, tgt_vocab, (batch_size, tgt_seq_len))

    src_mask = create_padding_mask(src)
    tgt_mask = create_causal_mask(tgt_seq_len).to(tgt.device) & create_padding_mask(tgt)

    output = model(src, tgt, src_mask, tgt_mask)
    print(output.shape) # expected output: (2, 4, 10000)

torch.Size([2, 4, 10000])


# 训练模型(en→fr)

In [11]:
# Install training dependencies
%pip -q install datasets tokenizers tqdm

Note: you may need to restart the kernel to use updated packages.


In [14]:
import random
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
from torch.utils.data import DataLoader

# Device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

# Reproducibility
seed = 42
random.seed(seed)
torch.manual_seed(seed)

# Load dataset
raw_ds = load_dataset("Helsinki-NLP/opus_books", "en-fr")
train_raw = raw_ds["train"]
if "validation" in raw_ds:
    valid_raw = raw_ds["validation"]
elif "test" in raw_ds:
    valid_raw = raw_ds["test"]
else:
    valid_raw = raw_ds["train"]

# Small subset to run quickly
MAX_TRAIN = 20000
MAX_VALID = 2000
train_raw = train_raw.select(range(min(MAX_TRAIN, len(train_raw))))
valid_raw = valid_raw.select(range(min(MAX_VALID, len(valid_raw))))

# Train a small BPE tokenizer
special_tokens = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]

tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

trainer = trainers.BpeTrainer(vocab_size=16000, special_tokens=special_tokens)

def batch_iterator():
    for ex in train_raw:
        yield ex["translation"]["en"]
        yield ex["translation"]["fr"]

tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)

pad_id = tokenizer.token_to_id("[PAD]")
bos_id = tokenizer.token_to_id("[BOS]")
eos_id = tokenizer.token_to_id("[EOS]")
vocab_size = tokenizer.get_vocab_size()

MAX_LEN = 128

def encode(text, max_len=MAX_LEN):
    ids = tokenizer.encode(text).ids
    ids = [bos_id] + ids[: max_len - 2] + [eos_id]
    return ids

def build_pairs(ds):
    pairs = []
    for ex in ds:
        src = encode(ex["translation"]["en"])
        tgt = encode(ex["translation"]["fr"])
        if len(src) <= MAX_LEN and len(tgt) <= MAX_LEN:
            pairs.append((src, tgt))
    return pairs

train_pairs = build_pairs(train_raw)
valid_pairs = build_pairs(valid_raw)
print("pairs:", len(train_pairs), len(valid_pairs))

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    max_src = max(len(x) for x in src_batch)
    max_tgt = max(len(x) for x in tgt_batch)

    src_tensor = torch.full((len(batch), max_src), pad_id, dtype=torch.long)
    tgt_tensor = torch.full((len(batch), max_tgt), pad_id, dtype=torch.long)

    for i, (s, t) in enumerate(zip(src_batch, tgt_batch)):
        src_tensor[i, : len(s)] = torch.tensor(s, dtype=torch.long)
        tgt_tensor[i, : len(t)] = torch.tensor(t, dtype=torch.long)

    return src_tensor, tgt_tensor

train_loader = DataLoader(train_pairs, batch_size=64, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_pairs, batch_size=64, shuffle=False, collate_fn=collate_fn)

device: mps



pairs: 20000 2000


In [15]:
from tqdm import tqdm

# Small model for quick run
model = Transformer(
    src_vocab=vocab_size,
    tgt_vocab=vocab_size,
    d_model=256,
    num_heads=4,
    num_layers=2,
    d_ff=512,
    dropout=0.1,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)


def run_epoch(loader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_tokens = 0

    with torch.set_grad_enabled(train):
        for src, tgt in tqdm(loader, leave=False):
            src = src.to(device)
            tgt = tgt.to(device)

            tgt_in = tgt[:, :-1]
            tgt_out = tgt[:, 1:]

            src_mask = create_padding_mask(src)
            tgt_mask = create_causal_mask(tgt_in.size(1)).to(device) & create_padding_mask(tgt_in)

            logits = model(src, tgt_in, src_mask, tgt_mask)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * tgt_out.numel()
            total_tokens += tgt_out.numel()

    return total_loss / max(1, total_tokens)


EPOCHS = 2
for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(valid_loader, train=False)
    print(f"epoch {epoch}: train loss={train_loss:.4f}, val loss={val_loss:.4f}")

epoch 1: train loss=6.4114, val loss=5.7746


epoch 2: train loss=5.2659, val loss=5.2656


In [16]:
def greedy_decode(src_text, max_len=64):
    model.eval()

    src = torch.tensor([encode(src_text)], device=device)
    src_mask = create_padding_mask(src)

    tgt = torch.tensor([[bos_id]], device=device)
    for _ in range(max_len):
        tgt_mask = create_causal_mask(tgt.size(1)).to(device) & create_padding_mask(tgt)
        logits = model(src, tgt, src_mask, tgt_mask)
        next_id = torch.argmax(logits[:, -1, :], dim=-1)
        tgt = torch.cat([tgt, next_id.unsqueeze(1)], dim=1)
        if next_id.item() == eos_id:
            break

    ids = tgt.squeeze(0).tolist()
    ids = [i for i in ids if i not in (bos_id, eos_id, pad_id)]
    return tokenizer.decode(ids)

print(greedy_decode("I love machine learning."))

– Et que je ne me dit - il n ' il me dit - il me dit - il me dit - il me dit - il me dit - il me dit - il me dit - il me dit - il me dit - il me dit - il me dit - il me dit - il me dit - il
